In [0]:
import os
import json

# 1. Paste your exact token structure here


# 2. This code automatically creates the file inside the Databricks cluster environment
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

# 3. Set the secure permissions required by Kaggle
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle token applied successfully!")

---------------------------------------------------------------------------
PermissionError                           Traceback (most recent call last)
File <command-5882197082039916>, line 8
      5 kaggle_credentials = {"username":"jyothish","key":"KGAT_4dd5c7c54c7118c066aeb51572429ed2"}
      7 # 2. This code automatically creates the file inside the Databricks cluster environment
----> 8 os.makedirs('/root/.kaggle', exist_ok=True)
      9 with open('/root/.kaggle/kaggle.json', 'w') as f:
     10     json.dump(kaggle_credentials, f)

File <frozen os>:225, in makedirs(name, mode, exist_ok)

PermissionError: [Errno 13] Permission denied: '/root/.kaggle'

In [0]:
import os
import json

# 1. Define a path inside the accessible /tmp/ directory
kaggle_dir = '/tmp/.kaggle'
os.makedirs(kaggle_dir, exist_ok=True)

# 2. Paste your exact token structure here
kaggle_credentials = {"username":"jyothish","key":"KGAT_4dd5c7c54c7118c066aeb51572429ed2"}

# 3. Write the json file into our new custom folder
token_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(token_path, 'w') as f:
    json.dump(kaggle_credentials, f)

# 4. Set the required strict permissions
os.chmod(token_path, 0o600)

# 5. CRITICAL: Tell Databricks/Kaggle to look in /tmp/.kaggle instead of /root/.kaggle
os.environ['KAGGLE_CONFIG_DIR'] = kaggle_dir

print("Kaggle token applied successfully in /tmp/!")

Kaggle token applied successfully in /tmp/!


In [0]:
%sh
# 1. Install/Update Kaggle
pip install kaggle --upgrade

# 2. Re-export the custom config directory path for this shell instance
export KAGGLE_CONFIG_DIR=/tmp/.kaggle

# 3. Create download destination folder
mkdir -p /tmp/nyc_taxi/

# 4. Run the download
kaggle datasets download -d elemento/nyc-yellow-taxi-trip-data -p /tmp/nyc_taxi/ --unzip

Dataset URL: https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data
License(s): U.S. Government Works


100%|██████████| 1.78G/1.78G [00:21<00:00, 88.7MB/s]


In [0]:
import os
# Check the contents of the target folder
print(os.listdir('/tmp/nyc_taxi/'))

['yellow_tripdata_2016-03.csv', 'yellow_tripdata_2016-02.csv', 'yellow_tripdata_2016-01.csv', 'yellow_tripdata_2015-01.csv']


In [0]:
import shutil
import os

# Define the source (/tmp) and the allowed destination (/Workspace)
src_dir = '/tmp/nyc_taxi'
dest_dir = '/Workspace/nyc_taxi_data'

# Create the folder in your Workspace
os.makedirs(dest_dir, exist_ok=True)

# Copy the CSV files over
for file_name in os.listdir(src_dir):
    if file_name.endswith('.csv'):
        shutil.copy(os.path.join(src_dir, file_name), os.path.join(dest_dir, file_name))

print("Files successfully moved to your /Workspace folder!")
print("Available files:", os.listdir(dest_dir))

Files successfully moved to your /Workspace folder!
Available files: ['yellow_tripdata_2016-03.csv', 'yellow_tripdata_2016-02.csv', 'yellow_tripdata_2016-01.csv', 'yellow_tripdata_2015-01.csv']


In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
-- create a custom catalog
CREATE CATALOG IF NOT EXISTS nyc_taxi_catalog;

In [0]:
# use the custom catalog name
chosen_catalog = "nyc_taxi_catalog" 

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {chosen_catalog}.nyc_taxi_db")
print(f"Schema created successfully in: {chosen_catalog}.nyc_taxi_db")

Schema created successfully in: nyc_taxi_catalog.nyc_taxi_db


In [0]:
# 1. Set up the schema in the default 'main' catalog
catalog_name = "nyc_taxi_catalog"
schema_name = "nyc_taxi_db"
schema_path = f"{catalog_name}.{schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_path}")

# 2. Files to process (now pointing to the workspace folder)
workspace_folder = "/Workspace/nyc_taxi_data"
files = ['yellow_tripdata_2015-01.csv', 'yellow_tripdata_2016-01.csv', 'yellow_tripdata_2016-02.csv', 'yellow_tripdata_2016-03.csv']

# 3. Read and Write Loop
for file_name in files:
    clean_table_name = "trips_" + file_name.replace("yellow_tripdata_", "").replace("-", "_").replace(".csv", "")
    full_table_path = f"{schema_path}.{clean_table_name}"
    
    print(f"Reading from Workspace: {file_name}...")
    
    # We load the file directly from the allowed Workspace path
    month_df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{workspace_folder}/{file_name}")
    
    print(f"Writing Managed Table to Unity Catalog -> {full_table_path}...")
    
    # Writing as a standard managed table in 'main' utilizes Databricks' 
    # pre-approved storage, completely bypassing the local file system restrictions.
    month_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_table_path)

print("\n🎉 Success! All tables have been securely written to the main catalog.")

Reading from Workspace: yellow_tripdata_2015-01.csv...
Writing Managed Table to Unity Catalog -> nyc_taxi_catalog.nyc_taxi_db.trips_2015_01...
Reading from Workspace: yellow_tripdata_2016-01.csv...
Writing Managed Table to Unity Catalog -> nyc_taxi_catalog.nyc_taxi_db.trips_2016_01...
Reading from Workspace: yellow_tripdata_2016-02.csv...
Writing Managed Table to Unity Catalog -> nyc_taxi_catalog.nyc_taxi_db.trips_2016_02...
Reading from Workspace: yellow_tripdata_2016-03.csv...
Writing Managed Table to Unity Catalog -> nyc_taxi_catalog.nyc_taxi_db.trips_2016_03...

🎉 Success! All tables have been securely written to the main catalog.


In [0]:
%sql
SELECT * FROM nyc_taxi_catalog.nyc_taxi_db.trips_2015_01 LIMIT 5;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
2,2015-01-23T18:54:13.000Z,2015-01-23T19:06:38.000Z,2,1.35,-74.00575256347656,40.718448638916016,1,N,-73.9906005859375,40.730777740478516,1,9.0,1.0,0.5,2.16,0.0,0.3,12.96
2,2015-01-23T18:54:13.000Z,2015-01-23T19:12:09.000Z,1,3.05,-73.97847747802734,40.76445388793945,1,N,-73.96755981445312,40.802642822265625,1,14.0,1.0,0.5,3.16,0.0,0.3,18.96
2,2015-01-23T18:54:13.000Z,2015-01-23T19:18:15.000Z,1,2.92,-73.98060607910156,40.7305908203125,1,N,-73.99107360839844,40.759002685546875,1,16.5,1.0,0.5,3.0,0.0,0.3,21.3
2,2015-01-23T18:54:13.000Z,2015-01-23T19:01:25.000Z,1,0.92,-73.9909896850586,40.76044845581055,1,N,-73.99089813232422,40.7504997253418,1,6.0,1.0,0.5,1.56,0.0,0.3,9.36
2,2015-01-23T18:54:13.000Z,2015-01-23T19:27:40.000Z,1,17.58,-73.7891616821289,40.64318084716797,2,N,-73.97000885009766,40.75668716430664,2,52.0,0.0,0.5,0.0,5.33,0.3,58.13
